In [1]:
import sys
import os

import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np
import warnings

warnings.filterwarnings("ignore")

In [2]:
import pandas as pd

In [3]:
dist_out_dir = '/home/workspace/spatial_mouse_lung_outputs/downstream_analysis/distance'
dist_out_dir_cellchat = '/home/workspace/spatial_mouse_lung_outputs/downstream_analysis/distance/cellchat_zone'

if not os.path.exists (dist_out_dir_cellchat):
    os.makedirs(dist_out_dir_cellchat)

plot_out_dir = os.path.join(dist_out_dir_cellchat, 'plots')
if not os.path.exists (plot_out_dir):
    os.makedirs(plot_out_dir)

In [4]:
adata = sc.read_h5ad(os.path.join(dist_out_dir,'adata_distance_zones_structure_tls_dist_categories.h5ad'))


In [5]:
adata.obs['label_fine'].unique().tolist()

['Col13a1+ fibroblast',
 'Alv Mf',
 'Cap',
 'Vein',
 'AT2',
 'Mono',
 'Th0',
 'Pericyte 2',
 'Cap-a',
 'Neut',
 'Pericyte 1',
 'Club',
 'Ciliated',
 'Art',
 'AT1',
 'CD4 naive',
 'B cell',
 'Th17',
 'Int Mf',
 'CD8 naive',
 'SMC',
 'gd T cell',
 'Plasmablast',
 'Th2',
 'Lymph',
 'Ccr7- cDC2',
 'NK cell',
 'cDC1',
 'CD4 trans',
 'Ccr7+ cDC2',
 'Th1',
 'CD8 act',
 'Mesothelial',
 'Treg',
 'Myofibroblast',
 'Col14a1+ fibroblast',
 'ILC2']

In [9]:
# Categorize by zone 
adata = adata[~adata.obs['zone_consol'].isna(), :]
# set labels
adata.obs['zone_consol'] = adata.obs['zone_consol'].values.tolist()
adata.obs['label_fine'] = adata.obs['label_fine'].values.tolist()

In [10]:
#wherever cell_type_1 is 'T Cells', add the classification to the cell_type_1 column
# Create a mask for CD4 act
cd4_mask = adata.obs['label_fine'].isin(['Th0', 'Th1', 'Th17', 'Th2', 'Treg', 'CD4 trans'])

# make single activated T cell label per region 
adata.obs.loc[cd4_mask, 'label_fine'] = (
    'CD4 act (' + adata.obs.loc[cd4_mask, 'zone_consol'] + ')'
)

# Check the updated cell types
print("Updated T cell categories:")
print(adata.obs.loc[cd4_mask, 'label_fine'].value_counts())



Updated T cell categories:
label_fine
CD4 act (parenchyma)    6313
CD4 act (adventitia)    2409
CD4 act (TLS)           1479
CD4 act (capsule)        201
CD4 act (vessels)         79
CD4 act (bronchi)          5
Name: count, dtype: int64


In [13]:
# Remove CD4 act in zones with low cell numbers 
# remove cells < 1% of population at day 3
celltypes_remove = ['CD4 act (capsule)', 'CD4 act (vessels)', 'CD4 act (bronchi)']
adata = adata[~adata.obs['label_fine'].isin(celltypes_remove), :]

In [14]:
# check full adata after removing these cell types
cd4_mask = adata.obs['label_fine'].isin(['CD4 act (parenchyma)', 'CD4 act (adventitia)', 'CD4 act (TLS)', 
                                         'CD4 act (capsule)', 'CD4 act (vessels)', 'CD4 act (bronchi)'])
print("Updated T cell categories:")
print(adata.obs.loc[cd4_mask, 'label_fine'].value_counts())

# check d3 adata
adata_d3 = adata[adata.obs['sample_label']=='HDM_day3', :]
cd4_mask_d3 = adata_d3.obs['label_fine'].isin(['CD4 act (parenchyma)', 'CD4 act (adventitia)', 'CD4 act (TLS)', 
                                         'CD4 act (capsule)', 'CD4 act (vessels)', 'CD4 act (bronchi)'])
print("Updated T cell categories:")
print(adata_d3.obs.loc[cd4_mask_d3, 'label_fine'].value_counts())

Updated T cell categories:
label_fine
CD4 act (parenchyma)    6313
CD4 act (adventitia)    2409
CD4 act (TLS)           1479
Name: count, dtype: int64
Updated T cell categories:
label_fine
CD4 act (parenchyma)    5918
CD4 act (adventitia)    1384
CD4 act (TLS)           1200
Name: count, dtype: int64


In [15]:
dist_out_dir_cellchat

'/home/workspace/spatial_mouse_lung_outputs/downstream_analysis/distance/cellchat_zone'

In [16]:
adata.write_h5ad(os.path.join(dist_out_dir_cellchat,'adata_cellchat_prepped.h5ad'))